# Differential RV recovery - line-by-line CCF + packaged ccfopt

Minimal E2E check of RV bias with the **packaged** static
`*.ccfopt.delta.mas` and the validated line-by-line kwargs in
:data:`~hispecdrp.utils.line_mask_utils.CCFOPT_SINGLE_LINE_STELLAR_RV_KW`
(`window_line_by_line=True`, `v_half_window=15000` m/s, `v_step=25` m/s).

Chunk RVs are the **Gaussian fit** on the CCF (no discrete-peak snap-back).

For the default **order-chunk** path (`window_size_um=0.25`), see
`rv_recovery_sim.ipynb`.

**Runtime (estimate):** about **1–3 hours** for the default 7-bulk grid
(`BULK_RV_KM_S` below: 14 full densify/UTR/Stage0-2 injections plus one
shared Stage-1 bundle).

> For each **bulk** in `BULK_RV_KM_S`, inject `(bulk, bulk + drv)` with
> `drv = 5 m/s`. Report absolute biases and the **matched-line** differential
> with joint inliers + hard QC. Require
> `|estimated_drv - drv| < 10 cm/s` at every bulk.

Physical path (`inject_rv_via='spectrum'`, fiber **2**):

1. Densify BT-Settl -> Doppler paint -> near-noiseless UTR -> Stage 0.
2. Stage 1: box extract (aperture 8) + Aidan WAVE + deg-0 order ZP and fiberflat/traces from the first injection
   (cached and reused for later bulks).
3. Stage 2: blaze; **sky_sub and barycorr skipped**; stellar RV with packaged
   ccfopt + `CCFOPT_SINGLE_LINE_STELLAR_RV_KW`.

Sequence:

1. For each bulk in `BULK_RV_KM_S` (default `-150...+150` km/s), inject
   `bulk` and `bulk + 5 m/s`.
2. Score matched-line `|estimated_drv - drv|` against 10 cm/s per bulk.

**Environment variables** (set before running):

| Variable | Required when | Purpose |
|---|---|---|
| `KOA_CALIBRATION_CACHE` | Always | Local KOA calibration cache |
| `KECK_OBSERVER_EMAIL` | Optional remote sync | Keck Observer email |
| `KECK_OBSERVER_PASSWORD` | Optional remote sync | Keck Observer password |

If `KOA_CALIBRATION_CACHE` is unset, the setup cell sets it under `OUT`.


In [1]:
import os

import hispecdrp

print(f"hispecdrp {hispecdrp.__version__} @ {hispecdrp.__file__}")

# Line-by-line products live under OUT; keep KOA cache shared with sibling runs.
OUT = "/Users/ckilpatrick/hispec_rv_recovery_sim/line_by_line"
os.makedirs(OUT, exist_ok=True)

os.environ.setdefault(
    "KOA_CALIBRATION_CACHE",
    "/Users/ckilpatrick/hispec_rv_recovery_sim/cals/KOA_CALIBRATION_CACHE",
)
print("output:", OUT)
print("KOA_CALIBRATION_CACHE:", os.environ["KOA_CALIBRATION_CACHE"])


hispecdrp 0.0.post2+waveinject.local @ /Users/ckilpatrick/Library/CloudStorage/Dropbox/scripts/python/cursor/HISPEC_DRP/hispecdrp/__init__.py
output: /Users/ckilpatrick/hispec_rv_recovery_sim/line_by_line
KOA_CALIBRATION_CACHE: /Users/ckilpatrick/hispec_rv_recovery_sim/cals/KOA_CALIBRATION_CACHE


## 1. Sync / verify KOA caldb (BSPEC)

Uses `ensure_koa_calibrations`: seed the notebook cache from another populated
KOA cache if needed, try remote sync, and fall back to local-only on auth /
network / API failures. After that, this notebook pins
`HISPEC_KOA_CONNECT_REMOTE=0` so Stage 0-2 lookups stay offline.


In [2]:
from hispecdrp.calibrations import ensure_koa_calibrations

SPECTROGRAPH = "BSPEC"

# Optional: pin an explicit seed cache (otherwise auto-discovers
# ~/hispec*/cals/KOA_CALIBRATION_CACHE and HISPEC_KOA_CACHE_SEED).
# KOA_SEED_SOURCES = ["/Users/ckilpatrick/hispec_rv_order_scan/cals/KOA_CALIBRATION_CACHE"]
KOA_SEED_SOURCES = None

summary = ensure_koa_calibrations(
    SPECTROGRAPH,
    seed_sources=KOA_SEED_SOURCES,
    ignore_missing_cal_types=["lfc_wavesol"],
)

# Keep the rest of this notebook offline: Stage 0/1 primitives must not
# re-login to KOA when the remote API/auth is down.
os.environ["HISPEC_KOA_CONNECT_REMOTE"] = "0"

seed = summary.get("seed") or {}
if seed.get("seeded"):
    print(f"seeded KOA cache from {seed['source']}  ({seed['n_fits_before']} -> {seed['n_fits_after']} FITS)")
elif seed.get("n_fits_before", 0):
    print(f"KOA cache already populated ({seed['n_fits_before']} FITS)")
print(f"remote_mode: {summary.get('remote_mode')}")
if summary.get("remote_error"):
    print(f"remote_error: {summary['remote_error']}")
print("HISPEC_KOA_CONNECT_REMOTE:", os.environ.get("HISPEC_KOA_CONNECT_REMOTE"))

print(
    f"{summary['spectrograph']} caldb: {summary['n_records']} record(s); "
    f"downloaded {summary['n_downloaded']}/{summary['n_missing']} missing file(s)"
)
print("types:", summary["cal_types"])
if summary["downloaded"]:
    print("downloaded:", summary["downloaded"])
if summary["failed"]:
    print("failed:", summary["failed"])
if summary.get("still_missing"):
    print("still_missing:", summary["still_missing"])


[2026-08-19T20:21:37][hispecdrp.calibrations.koa_sync:353][INFO]: KOA cache already populated (16 FITS) at /Users/ckilpatrick/hispec_rv_recovery_sim/cals/KOA_CALIBRATION_CACHE; skip seed.
[2026-08-19T20:21:37][koa_middleware.store:186][INFO]: Cache initialized: cache_dir='/Users/ckilpatrick/hispec_rv_recovery_sim/cals/KOA_CALIBRATION_CACHE', data_dir='/Users/ckilpatrick/hispec_rv_recovery_sim/cals/KOA_CALIBRATION_CACHE/calibrations/hispec/', db='/Users/ckilpatrick/hispec_rv_recovery_sim/cals/KOA_CALIBRATION_CACHE/database/hispec_calibrations.db', table='hispec'
[2026-08-19T20:21:37][koa_middleware.keck_client:81][INFO]: Session validation: no observer ID found; re-login required.
[2026-08-19T20:21:37][koa_middleware.keck_client:51][INFO]: No valid login detected, logging in...
[2026-08-19T20:21:39][koa_middleware.keck_client:136][INFO]: User info request successful: observer ID=4187
[2026-08-19T20:21:39][koa_middleware.keck_client:56][INFO]: Keck Observer login successful! Observer ID 

## 2. Injection configuration

Fiber 2. Fixed Aidan WAVE + deg-0 order ZP. Spectrum Doppler paint only.
Near-noiseless UTR. Uses the packaged static `ccfopt.delta.mas` (**not**
rebuilt here) with **line-by-line** stellar RV
(`window_line_by_line=True`, `CCFOPT_SINGLE_LINE_STELLAR_RV_KW`).

Edit **`BULK_RV_KM_S`** below to choose bulk offsets. Default sweeps
`-150...+150` km/s in 50 km/s steps. For each bulk, injects are
`(bulk)` and `(bulk + 5 m/s)`.


In [3]:
from pathlib import Path

from hispecdrp.utils.line_mask_utils import (
    CCFOPT_SINGLE_LINE_STELLAR_RV_KW,
    MaskLines,
    packaged_default_line_mask_path,
    read_mask_generation_meta,
)

FIBERS = [2]
INJECT_RV_VIA = "spectrum"  # densify -> Doppler paint; WAVE fixed (Aidan + ZP0)
EXTRACTION_APERTURE = 8
TEMPLATE = "lte049.0-2.5-0.0a+0.0.BT-Settl.spec.7.h5"
# Packaged default mask (same as resolve_mask_path(None)); do not overwrite here.
MASK_PATH = packaged_default_line_mask_path()

POISSON_NOISE = False
CONVERT_TO_UINT16 = False

SEED = 0
DRV_M_S = 5.0
REQUIREMENT_M_S = 0.10  # 10 cm/s

# Bulk RV offsets (km/s). Edit this list to run a subset or different grid.
# For each bulk B, injects are B and B + DRV_M_S (m/s).
BULK_RV_KM_S = [-150.0, -100.0, -50.0, 0.0, 50.0, 100.0, 150.0]


def bulk_run_label(bulk_km_s: float, which: str) -> str:
    """Stable run-dir label for a bulk / epoch (a=bulk, b=bulk+drv)."""
    sign = "m" if bulk_km_s < 0 else "p"
    return f"bulk_{sign}{abs(bulk_km_s):06.1f}kms_{which}".replace(".", "p")


def stellar_rv_prim_args(ccf_input_velocity_m_s: float, *, save_preview: bool = False) -> dict:
    """Line-by-line ccf_line_mask matching CCFOPT_SINGLE_LINE_STELLAR_RV_KW."""
    kw = CCFOPT_SINGLE_LINE_STELLAR_RV_KW
    return {
        "algorithm": "ccf_line_mask",
        "mask_path": MASK_PATH,
        "ccf_input_velocity": float(ccf_input_velocity_m_s),
        "do_fibers": list(FIBERS),
        "window_line_by_line": True,
        "window_size_um": float(kw["window_size_um"]),
        "v_half_window": float(kw["v_half_window_m_s"]),
        "v_step_coarse": float(kw["v_step_m_s"]),
        "mask_auto_center_scout": bool(kw["mask_auto_center_scout"]),
        "mask_ccf_pixel_interp_order": int(kw["mask_ccf_pixel_interp_order"]),
        "ccf_min_good_pixels": int(kw["ccf_min_good_pixels"]),
        # Sim: BERV forced 0; keep CCF zb identically 0.
        "ccf_zb": 0.0,
        "ccf_zb_from_barycorr": False,
        "save_preview": bool(save_preview),
        "preview_write_png": False,
        "show_preview": False,
    }


assert Path(MASK_PATH).is_file(), f"missing packaged mask: {MASK_PATH}"
_mask_meta = read_mask_generation_meta(MASK_PATH) or {}
_n_mask = len(MaskLines.load(MASK_PATH).line_lambda_um)
_n_inj = 2 * len(BULK_RV_KM_S)
_srv = stellar_rv_prim_args(0.0)
print(f"fiber(s): {FIBERS}")
print(f"extraction_aperture = {EXTRACTION_APERTURE}")
print(f"inject_rv_via = {INJECT_RV_VIA!r}  (densify then Doppler only at ZEMAX)")
print(f"poisson_noise={POISSON_NOISE}  convert_to_uint16={CONVERT_TO_UINT16}")
print(f"mask_path = {MASK_PATH}")
print(
    f"mask n_lines={_n_mask}  "
    f"generator={_mask_meta.get('generator')}"
)
print(
    f"stellar RV: line-by-line  "
    f"window_line_by_line={_srv['window_line_by_line']}  "
    f"window_size_um={_srv['window_size_um']}  "
    f"v_half_window={_srv['v_half_window']}  "
    f"v_step_coarse={_srv['v_step_coarse']}  "
    f"mask_ccf_pixel_interp_order={_srv['mask_ccf_pixel_interp_order']}  "
)
print(f"seed={SEED}  drv={DRV_M_S:.3f} m/s")
print(f"BULK_RV_KM_S = {BULK_RV_KM_S}")
print(
    f"per bulk: inject bulk and bulk+{DRV_M_S:.3f} m/s  "
    f"({_n_inj} injections total)"
)
print(
    f"runtime estimate: ~1-3 h for {_n_inj} full E2E injections "
    f"(scales roughly with len(BULK_RV_KM_S)={len(BULK_RV_KM_S)}; "
    f"line-by-line CCF is slower than order-chunk)"
)
print(f"requirement: |estimated_drv - drv| < {REQUIREMENT_M_S:.2f} m/s (10 cm/s)")


## 3. Injection helper (lazy shared Stage-1 cals)

Defines `run_one_injection`. The **first** science call builds a complete Stage-1
bundle from that injection's rate map (fiberflat / traces / Aidan+deg-0 WAVE) via
`ensure_shared_stage1_cals_for_rv_sim` and caches it under `OUT/_shared_stage1`.
Later bulks (e.g. `bulk+5 m/s`) reuse the same frames. Stage-2 skips sky_sub and
barycorr; stellar RV uses line-by-line `ccf_line_mask` via `stellar_rv_prim_args`.


In [4]:
from pathlib import Path
import importlib

from hispecdrp.pipelines import Stage0Pipeline, Stage1Pipeline, Stage2Pipeline
from hispecdrp.simulations import e2e_pipeline as _e2e_pipeline

# Long-lived kernels may still hold a pre-helper copy of e2e_pipeline in
# sys.modules; reload so ensure_shared_stage1_cals_for_rv_sim is visible.
importlib.reload(_e2e_pipeline)
from hispecdrp.simulations.e2e_pipeline import ensure_shared_stage1_cals_for_rv_sim
from hispecdrp.simulations.simulate_up_the_ramp import create_ramp_from_spectrum

# Lazily filled on the first science injection (or reloaded from disk).
# Complete HISPEC Stage-1 bundle: tracepos, traceprof, fiberflat, Aidan+ZP0 WAVE.
SHARED_CALS = None
SHARED_DIR = str(Path(OUT) / "_shared_stage1")
REBUILD_SHARED_CALS = False  # set True to force rebuild from the next injection


def run_one_injection(
    label: str,
    injected_rv_km_s: float,
    seed: int,
    *,
    save_preview: bool = False,
) -> dict:
    """Densify→Doppler→ZEMAX UTR, Stage0-2; share Stage-1 cals across the grid.

    The first call (or a cache miss under ``SHARED_DIR``) builds a complete
    calibration bundle from this injection's rate map: fiberflat / traces /
    Aidan+deg-0 order-ZP WAVE. Later injections reuse those same frames.
    """
    global SHARED_CALS

    run_dir = str(Path(OUT) / label)
    os.makedirs(run_dir, exist_ok=True)
    injected_m_s = float(injected_rv_km_s) * 1000.0

    print(f"\n=== {label}: inject {injected_rv_km_s:.6f} km/s, seed={seed} ===")
    # Science ramp only — Stage-1 masters come from SHARED_CALS (built once).
    ramp, _cals, rate_map = create_ramp_from_spectrum(
        run_dir,
        spectrograph=SPECTROGRAPH,
        fibers=FIBERS,
        template_filename=TEMPLATE,
        mag_ab=3.05,
        filter_band="J",
        seed=seed,
        radial_velocity_km_s=injected_rv_km_s,
        barycentric_velocity_km_s=0.0,
        inject_rv_via=INJECT_RV_VIA,
        poisson_noise=POISSON_NOISE,
        convert_to_uint16=CONVERT_TO_UINT16,
        register_stage1_sims=False,
        register_wavesol=False,
    )
    print(f"ramp={ramp.meta.filename}  shape={ramp.data.shape}")

    if SHARED_CALS is None or REBUILD_SHARED_CALS:
        SHARED_CALS = ensure_shared_stage1_cals_for_rv_sim(
            SHARED_DIR,
            rate_map=rate_map,
            spectrograph=SPECTROGRAPH,
            fibers=FIBERS,
            known_rv_m_s=injected_m_s,
            extraction_aperture=float(EXTRACTION_APERTURE),
            ramp_meta=getattr(ramp, "meta", None),
            rebuild=bool(REBUILD_SHARED_CALS),
        )
        zp = SHARED_CALS.get("zp_info") or {}
        print(
            f"SHARED_CALS ready under {SHARED_DIR}  "
            f"note={getattr(SHARED_CALS['wavesol'].meta, 'note', None)}  "
            f"median order ZP = {zp.get('median_dv_m_s', float('nan')):.3f} m/s  "
            f"(from_cache={zp.get('loaded_from_cache')})"
        )

    do_fibers = [str(f) for f in FIBERS]
    l0 = Stage0Pipeline(output_dir=run_dir).apply(ramp)
    l1 = Stage1Pipeline(
        output_dir=run_dir,
        prim_args={
            "spec_extract": {
                "do_fibers": do_fibers,
                "method": "box",
                "extraction_aperture": int(EXTRACTION_APERTURE),
                "tracepos": SHARED_CALS["trace_pos"],
                "traceprof": SHARED_CALS["trace_prof"],
            },
            "set_wavesol": {"wavesol": SHARED_CALS["wavesol"]},
        },
    ).apply(l0)
    l2 = Stage2Pipeline(
        output_dir=run_dir,
        prim_args={
            "blaze_corr": {
                "do_fibers": do_fibers,
                "fiberflat": SHARED_CALS["fiberflat"],
            },
            # No sky fiber in these single-science-fiber sims; skip sky subtraction.
            "sky_sub": {"run": False},
            # BERV forced 0 at sim time; skip barycorr so CCF zb stays identically 0.
            "barycorr": {"run": False},
            "stellar_rv": stellar_rv_prim_args(
                injected_m_s, save_preview=bool(save_preview)
            ),
        },
    ).apply(l1)

    fib = FIBERS[0]
    avg = getattr(l2, f"rvccfavg_fib{fib}", None)
    if avg is None or len(avg) == 0:
        raise RuntimeError(f"{label}: missing rvccfavg_fib{fib}")
    rv_rec = float(avg["RV"][0])
    rv_err = float(avg["RV_ERR"][0])
    preview_html = None
    if save_preview:
        preview_html = l2.get_output_path(
            output_dir=run_dir, suffix="stellar_rv"
        ).replace(".fits", "_preview.html")
        print(f"stellar RV preview HTML: {preview_html}")
    print(
        f"fiber {fib}: recovered RV = {rv_rec:.9f} +/- {rv_err:.9f} m/s  "
        f"(bias vs inject = {rv_rec - injected_m_s:.9f} m/s)"
    )
    return {
        "label": label,
        "run_dir": run_dir,
        "seed": seed,
        "inject_km_s": float(injected_rv_km_s),
        "inject_m_s": injected_m_s,
        "recover_m_s": rv_rec,
        "recover_err_m_s": rv_err,
        "wavesol": SHARED_CALS["wavesol"],
        "cals": SHARED_CALS,
        "preview_html": preview_html,
        "l2": l2,
    }


## 4. Inject each bulk pair (line-by-line stellar RV; no mask rebuild)


In [5]:
import numpy as np

RESULTS = {}  # bulk_km_s -> {"a": result, "b": result}
drv_km_s = DRV_M_S / 1000.0

for i_bulk, bulk_km_s in enumerate(BULK_RV_KM_S):
    bulk = float(bulk_km_s)
    lab_a = bulk_run_label(bulk, "a")
    lab_b = bulk_run_label(bulk, "b")
    save_preview = i_bulk == 0
    result_a = run_one_injection(lab_a, bulk, SEED, save_preview=save_preview)
    result_b = run_one_injection(
        lab_b, bulk + drv_km_s, SEED, save_preview=False
    )
    RESULTS[bulk] = {"a": result_a, "b": result_b}

# Sanity: WAVE arrays should match across injections (fixed WAVE path)
_wave_attr = f"wave_fib{FIBERS[0]}"
_first = RESULTS[float(BULK_RV_KM_S[0])]
w_a = np.asarray(getattr(_first["a"]["wavesol"], _wave_attr), dtype=np.float64)
w_b = np.asarray(getattr(_first["b"]["wavesol"], _wave_attr), dtype=np.float64)
print(
    f"\nmax |WAVE fib{FIBERS[0]} (B - A)| = {np.nanmax(np.abs(w_b - w_a)):.3e}  "
    f"(should be ~0 for inject_rv_via='spectrum')"
)

print("\n=== Absolute recovery / bias (line-by-line stellar RV) ===")
for bulk in BULK_RV_KM_S:
    for which in ("a", "b"):
        r = RESULTS[float(bulk)][which]
        print(
            f"{r['label']:28s} inject = {r['inject_m_s']:12.3f} m/s  "
            f"recover = {r['recover_m_s']:.9f} +/- {r['recover_err_m_s']:.9f} m/s  "
            f"bias = {r['recover_m_s'] - r['inject_m_s']:.9f} m/s"
        )



=== bulk_m0150p0kms_a: inject -150.000000 km/s, seed=0 ===
[2026-08-19T20:21:42][hispecdrp.simulations.simulate_up_the_ramp:1620][INFO]: inject_rv_via='spectrum': densify→Doppler-paint radial_velocity_km_s=-150.000000; WAVE left at packaged line-ZP bake
[2026-08-19T20:21:42][hispecdrp.simulations.simulate_up_the_ramp:1309][INFO]: Building default rate map (fibers=[2], spectrograph=BSPEC, output_dir=/Users/ckilpatrick/hispec_rv_recovery_sim/line_by_line/bulk_m0150p0kms_a).
[2026-08-19T20:21:43][hispecdrp.primitives.utils.spectral_utils:493][INFO]: Normalized spectrum to J=3.05 (AB mag); scale=5.8159e+04 (was 14.96 mag)
[2026-08-19T20:21:43][hispecdrp.simulations.simulate_up_the_ramp:1368][INFO]: Densifying fiber=2 spectrum to R>=3e5 before Doppler (PCHIP, densify-only)
[2026-08-19T20:21:44][hispecdrp.simulations.simulate_up_the_ramp:1385][INFO]: Applying densified-template Doppler shift (topocentric) radial_velocity_km_s=-150.000000
[2026-08-19T20:21:45][hispecdrp.simulations.simulate_

/Users/ckilpatrick/scripts/anaconda3/envs/hispec/lib/python3.13/site-packages/erfa/core.py:133: ErfaWarning: ERFA function "dtf2d" yielded 1 of "dubious year (Note 6)"
  warn(f'ERFA function "{func_name}" yielded {wmsg}', ErfaWarning)


ramp=simulated_ramp.fits  shape=(10, 4096, 4160)
[2026-08-19T20:22:45][hispecdrp.simulations.e2e_pipeline:2254][INFO]: Reusing shared Stage-1 cals under /Users/ckilpatrick/hispec_rv_recovery_sim/line_by_line/_shared_stage1/cals (tracepos/traceprof/fiberflat + Aidan WAVE).
SHARED_CALS ready under /Users/ckilpatrick/hispec_rv_recovery_sim/line_by_line/_shared_stage1  note=aidan_gws+order_zp0  median order ZP = nan m/s  (from_cache=True)
[2026-08-19T20:22:45][hispecdrp.pipelines.base_pipeline:130][INFO]: Applying Stage0Pipeline with arguments:
input = RampModel('simulated_ramp.fits')
run = True
save_result = True
output_model = input
output_file = None
output_dir = /Users/ckilpatrick/hispec_rv_recovery_sim/line_by_line/bulk_m0150p0kms_a
suffix = None
save_preview = False
show_preview = False
[2026-08-19T20:22:45][hispecdrp.primitives.base_primitive:282][INFO]: Applying DQInitPrimitive with arguments:
input = RampModel('simulated_ramp.fits')
run = True
save_result = None
output_model = inp

/Users/ckilpatrick/scripts/anaconda3/envs/hispec/lib/python3.13/site-packages/erfa/core.py:133: ErfaWarning: ERFA function "d2dtf" yielded 1 of "dubious year (Note 5)"
  warn(f'ERFA function "{func_name}" yielded {wmsg}', ErfaWarning)


/Users/ckilpatrick/scripts/anaconda3/envs/hispec/lib/python3.13/site-packages/erfa/core.py:133: ErfaWarning: ERFA function "dtf2d" yielded 1 of "dubious year (Note 6)"
  warn(f'ERFA function "{func_name}" yielded {wmsg}', ErfaWarning)


/Users/ckilpatrick/scripts/anaconda3/envs/hispec/lib/python3.13/site-packages/erfa/core.py:133: ErfaWarning: ERFA function "dtf2d" yielded 1 of "dubious year (Note 6)"
  warn(f'ERFA function "{func_name}" yielded {wmsg}', ErfaWarning)


... (7 outputs left out)

/Users/ckilpatrick/scripts/anaconda3/envs/hispec/lib/python3.13/site-packages/erfa/core.py:133: ErfaWarning: ERFA function "dtf2d" yielded 1 of "dubious year (Note 6)"
  warn(f'ERFA function "{func_name}" yielded {wmsg}', ErfaWarning)


/Users/ckilpatrick/scripts/anaconda3/envs/hispec/lib/python3.13/site-packages/erfa/core.py:133: ErfaWarning: ERFA function "dtf2d" yielded 1 of "dubious year (Note 6)"
  warn(f'ERFA function "{func_name}" yielded {wmsg}', ErfaWarning)


/Users/ckilpatrick/scripts/anaconda3/envs/hispec/lib/python3.13/site-packages/erfa/core.py:133: ErfaWarning: ERFA function "dtf2d" yielded 1 of "dubious year (Note 6)"
  warn(f'ERFA function "{func_name}" yielded {wmsg}', ErfaWarning)


/Users/ckilpatrick/scripts/anaconda3/envs/hispec/lib/python3.13/site-packages/erfa/core.py:133: ErfaWarning: ERFA function "dtf2d" yielded 1 of "dubious year (Note 6)"
  warn(f'ERFA function "{func_name}" yielded {wmsg}', ErfaWarning)


## 5. Differential score vs 10 cm/s (matched-line, per bulk)

Uses :func:`~hispecdrp.primitives.stellar_rv.rv_differential_utils.estimate_matched_line_differential_rv`
(joint inliers + hard QC + leverage caps; default `max_rv_err_m_s=10`).
Expect residuals at the ~1 m/s level with the packaged ccfopt mask; the
strict 10 cm/s assert may still fail on fresh E2E products.


In [6]:
from hispecdrp.primitives.stellar_rv.rv_differential_utils import (
    estimate_matched_line_differential_rv,
)


def _score_pair_matched(name: str, a: dict, b: dict) -> dict:
    """Score drv with matched-line delta + joint inliers."""
    drv = b["inject_m_s"] - a["inject_m_s"]
    fib = FIBERS[0]
    ta = getattr(a["l2"], f"rvccf_fib{fib}")
    tb = getattr(b["l2"], f"rvccf_fib{fib}")
    diff = estimate_matched_line_differential_rv(
        ta,
        tb,
        inject_a_m_s=float(a["inject_m_s"]),
        inject_b_m_s=float(b["inject_m_s"]),
    )
    estimated_drv = float(diff["estimated_drv_m_s"])
    residual = float(diff["residual_m_s"])
    passed = bool(np.isfinite(residual) and abs(residual) < REQUIREMENT_M_S)

    print(f"=== Differential {name} (matched-line) ===")
    print(
        f"  A inject = {a['inject_m_s']:.9f} m/s  recover = {a['recover_m_s']:.9f}  "
        f"bias = {a['recover_m_s'] - a['inject_m_s']:.9f} m/s"
    )
    print(
        f"  B inject = {b['inject_m_s']:.9f} m/s  recover = {b['recover_m_s']:.9f}  "
        f"bias = {b['recover_m_s'] - b['inject_m_s']:.9f} m/s"
    )
    print(f"  drv                 = {drv:.12f} m/s")
    print(
        f"  estimated_drv (matched-line) = {estimated_drv:.12f} +/- "
        f"{diff['estimated_drv_err_m_s']:.12f} m/s  "
        f"(n_kept={diff['n_kept']}/{diff['n_matched']})"
    )
    print(
        f"  estimated_drv - drv = {residual:.12f} m/s  "
        f"({residual * 100.0:.9f} cm/s)"
    )
    print(
        f"  requirement         = |estimated_drv - drv| < {REQUIREMENT_M_S:.2f} m/s "
        f"-> {'PASS' if passed else 'FAIL'}"
    )
    if not passed:
        print(
            f"  WARNING: residual {residual:.12f} m/s exceeds {REQUIREMENT_M_S:.2f} m/s"
        )
    return {
        "name": name,
        "bulk_km_s": float(a["inject_km_s"]),
        "drv": drv,
        "estimated_drv": estimated_drv,
        "estimated_drv_err": float(diff["estimated_drv_err_m_s"]),
        "residual": residual,
        "residual_cm_s": residual * 100.0,
        "passed": passed,
        "n_kept": int(diff["n_kept"]),
        "n_matched": int(diff["n_matched"]),
    }


SCORES = []
for bulk in BULK_RV_KM_S:
    pair = RESULTS[float(bulk)]
    name = f"bulk={bulk:+.1f} km/s (drv={DRV_M_S:.0f} m/s)"
    SCORES.append(_score_pair_matched(name, pair["a"], pair["b"]))
    print()


def _fmt_bulk_km_s(bulk: float) -> str:
    if abs(float(bulk)) < 0.5:
        return "0"
    return f"{float(bulk):+.0f}"


def _fmt_resid_cm_s(resid: float) -> str:
    return f"{float(resid):+.2f}"


import pandas as pd
from IPython.display import display

summary = pd.DataFrame(
    [
        {
            "bulk (km/s)": _fmt_bulk_km_s(s["bulk_km_s"]),
            "residual (cm/s)": _fmt_resid_cm_s(s["residual_cm_s"]),
            "n_chunks_kept": f"{s['n_kept']}/{s['n_matched']}",
            "result": "PASS" if s["passed"] else "FAIL",
        }
        for s in SCORES
    ]
)
display(summary)

n_pass = sum(1 for s in SCORES if s["passed"])
print(f"Overall: {n_pass}/{len(SCORES)} PASS")
assert n_pass == len(SCORES), (
    f"{len(SCORES) - n_pass} bulk(s) failed |estimated_drv - drv| < {REQUIREMENT_M_S} m/s"
)


=== Differential bulk=-150.0 km/s (drv=5 m/s) (matched-line) ===
  A inject = -150000.000000000 m/s  recover = -149996.916261560  bias = 3.083738440 m/s
  B inject = -149995.000000000 m/s  recover = -149991.774497567  bias = 3.225502433 m/s
  drv                 = 5.000000000000 m/s
  estimated_drv (matched-line) = 4.945675588498 +/- 0.946919155802 m/s  (n_kept=52/339)
  estimated_drv - drv = -0.054324411502 m/s  (-5.432441150 cm/s)
  requirement         = |estimated_drv - drv| < 0.10 m/s -> PASS

=== Differential bulk=-100.0 km/s (drv=5 m/s) (matched-line) ===
  A inject = -100000.000000000 m/s  recover = -99998.840415256  bias = 1.159584744 m/s
  B inject = -99995.000000000 m/s  recover = -99993.795392911  bias = 1.204607089 m/s
  drv                 = 5.000000000000 m/s
  estimated_drv (matched-line) = 5.018229926410 +/- 0.821472991241 m/s  (n_kept=65/334)
  estimated_drv - drv = 0.018229926410 m/s  (1.822992641 cm/s)
  requirement         = |estimated_drv - drv| < 0.10 m/s -> PASS


,bulk (km/s),residual (cm/s),n_chunks_kept,result
0,-150,-5.43,52/339,PASS
1,-100,+1.82,65/334,PASS
2,-50,+0.08,70/335,PASS
3,0,-0.91,63/335,PASS
4,+50,-1.11,61/338,PASS
5,+100,-1.41,63/338,PASS
6,+150,+1.08,64/343,PASS


Overall: 7/7 PASS
